In [ ]:
import sys
import os
import os
import phydrus as ps
import numpy as np
import warnings
import pandas as pd 
import os
import matplotlib.pyplot as plt
warnings.simplefilter(action='ignore', category=FutureWarning)
%matplotlib inline
current_dir = os.getcwd()

parent_dir = os.path.dirname(current_dir)
sys.path.append(parent_dir)

from model_build import InfiltrationModel

In [ ]:

pond_max= 3.0
#flux = np.zeros(22, dtype=np.float64)
discretize = {"layers": [49, 51], "dz": [1.0, 1.0]}

flux = np.array([ -1.,-1.,-4,-4,-4,-2.,-2.,0.,0.,0.,1.,1., 1.,1.,1.,1.,1.,1.,1.,1.]) / 1440 
temp_time  = 1440.0
sim_time = temp_time * flux.shape[0]
flux_bot = np.zeros(flux.shape)
head_bot,head_top = np.zeros(flux.shape),np.ones(flux.shape)
trans = np.array([0.0] * (flux.shape[0]),dtype=np.float64) # if root wateruptake is active transpiration needs to be given 
#bound_bot = 'seepage face'
bound_bot = 'free drainage'
bound_top = 'atmospheric'
soil_props = {"tr":[0.065, 0.095],"ths":[0.41, 0.41],"a":[0.075, 0.019],"n": [1.89, 1.31], "m":[1 - (1 / 1.89),1 - (1 / 1.31)],"ks":[106.1 / 1440, 6.24 / 1440], 
"L":[0.5, 0.5]}
hyd_model = 'VGM'
model = InfiltrationModel(sim_time,temp_time,discretize) #create the model!
hini = np.ones(model.nodes,dtype=np.float64) * -100
model.set_soil_model(hyd_model,soil_props)
model.set_boundary_conditions(hini,bound_top,bound_bot,pond_max)
hout = model.set_run_solver(hini,flux,flux_bot,head_top,head_bot,trans)
print(hout[:,-1])

In [ ]:
# Folder for Hydrus files to be stored
ws = "example_2"
# Path to folder containing hydrus.exe 
exe = os.path.join('/Users/onursahin/Flux1Dpy/hydrus_source/source_code/source/hydrus')
# Description
desc = "Infiltration of Water into a Two-Layered Soil Profile"
# Create model
ml = ps.Model(exe_name=exe, ws_name=ws, name="model", description=desc, 
              mass_units="mmol", time_unit="days", length_unit="cm")
ml.basic_info["lFlux"] = True
ml.basic_info["lShort"] = False

time = np.arange(1,20,1)

ml.add_time_info(tmax=20, print_array=time, dt=0.001, dtmax=0.1)
ml.add_waterflow(model=0, top_bc=2, bot_bc=4,tolh=1,tolth=10**-4)
m = ml.get_empty_material_df(n=2)
m.loc[[1, 2]] = [[0.095, 0.41, 0.019, 1.31, 6.24, 0.5],
                 [0.065, 0.41, 0.075, 1.89, 106.1, 0.5]]
ml.add_material(m)
nodes = 100  # Disctretize soil column into n elements
depth = [-51, -100]  # Depth of the soil column
ihead = -100.0  # Determine initial Pressure Head
# Create Profile
profile = ps.create_profile(bot=depth, dx=abs(depth[-1] / nodes), h=ihead, mat=m.index)
ml.add_profile(profile)  # Add the profile
# Add observation nodes at depth
ml.add_obs_nodes([0, -50, -100])
time = (2, 5, 7, 10, 20)
bc = {"tAtm": time, "Prec": (2, 5.5, 2, 0, 0), "rSoil": (0, 0, 0, 0, 1)}
#bc = {"tAtm": time, "Prec": (0, 0, 0, 0, 0), "rSoil": (0.1, 0.1, 0.1, 0.1, 0.1)}
atm = pd.DataFrame(bc, index=time)

ml.add_atmospheric_bc(atm, hcrits=3.0, hcrita=50000)
ml.write_input()
ml.simulate()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
import warnings

# Silence the Pandas deprecation warning from phydrus
warnings.simplefilter(action='ignore', category=FutureWarning)

# 1. Coordinate System Alignment
# ---------------------------------------------------------
# CUSTOM MODEL (Positive Upward):
# Assuming your code maps z = 0 (bottom) to z = 100 (surface), 
# and hout array index 0 is the BOTTOM and index 100 is the TOP.
# We shift this so the surface is 0 and the bottom is -100 for plotting.
z_custom = np.linspace(-100, 0, 101) 

# Note: If your hout array has index 0 as the TOP and index 100 as the BOTTOM, 
# simply change the above line back to: z_custom = np.linspace(0, -100, 101)
# ---------------------------------------------------------

hydrus_res = ml.read_nod_inf()
hydrus_times = sorted(list(hydrus_res.keys()))

fig, ax = plt.subplots(figsize=(6, 6))
colors = plt.cm.viridis(np.linspace(0, 1, len(hydrus_times)))

i,t = 0 , 4





# --- Plot Hydrus Model (Negative Upward) ---
df_t = hydrus_res[t]
hydrus_head = df_t['Moisture']
custom_head = hout[t, :]

# Force Hydrus depths to map from 0 (surface) down to -100 (bottom)
# This prevents plotting errors if Hydrus outputted them as positive depths
y_hydrus = -np.abs(df_t['Depth'])

ax.plot(hydrus_head, y_hydrus, 
        color='green', 
        marker='o',
        markersize=4,
        markevery=5,
        label='Hydrus' if i == len(hydrus_times)-1 else "")



print(np.max(np.abs(custom_head - hydrus_head[::-1])))
ax.plot(custom_head, z_custom[:len(custom_head)], 
        color='blue', 
        linestyle='-', 
        linewidth=2,
        label='New Model' if i == len(hydrus_times)-1 else "")

# Aesthetics and Labels
ax.set_title("Pressure Head Profile Comparison: Custom vs Hydrus", fontsize=14, pad=15)
ax.set_xlabel("Pressure Head (cm)", fontsize=12)
ax.set_ylabel("Elevation / Depth from Surface (cm)", fontsize=12)

# Ensure the y-axis strictly views the surface at the top and bottom at the bottom

    

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import warnings

# Silence the pandas deprecation warning
warnings.simplefilter(action='ignore', category=FutureWarning)

# 1. Extract Hydrus Data (Specific Print Times)
hydrus_res = (ml.read_nod_inf())

hydrus_times = sorted(list(hydrus_res.keys())) # e.g., [0.0, 2.0, 4.0, 5.0, 6.0, 10.0, 20.0]

plt.figure(figsize=(8, 6))


t = 11

print(t)
df_t = hydrus_res[t]
hcol = df_t['head'] if 'head' in df_t.columns else df_t['Head'][::-1].values
hnew = hout[t, :]
print(hnew[-10:],hcol[-10:])
print(hnew[0:10],hcol[0:10])
print()

